<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_rh_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *TelcomCI RH Analytics* dans Power BI Desktop.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| 🎓 | Une méthode opérationnelle pour construire un visuel précis |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |

### Le contexte métier

**TelcomCI** est un opérateur télécom ivoirien de 506 employés répartis dans 10 départements. Le dashboard répond à 5 questions :

| Page | Question |
|---|---|
| 1 — Vue DRH | Quelle est la santé RH globale ce trimestre ? |
| 2 — Masse Salariale | Comment évolue la paie et où se concentre-t-elle ? |
| 3 — Turnover | Qui part, où, à quel coût ? |
| 4 — Absentéisme | Quels départements / mois alertent ? |
| 5 — Performance | Quels top performers sont sous-payés et risquent de partir ? |

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

| Fichier | Grain | Rôle | Volumétrie |
|---|---|---|---|
| `departements.csv` | 1 département (avec sa cible d'effectif) | Dimension | 10 lignes |
| `vw_employes_propres.csv` | 1 employé propre (vue préparée) | Dimension principale | 506 lignes |
| `vw_salaires_valides.csv` | 1 fiche de paie d'1 employé × 1 mois | Fait | ~17 000 lignes |
| `vw_absences_valides.csv` | 1 épisode d'absence d'1 employé | Fait | ~1 200 lignes |
| `evaluations.csv` | 1 évaluation annuelle d'1 employé | Fait | 440 lignes |
| `recrutements.csv` | 1 campagne de recrutement | Fait | ~75 lignes |

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

### Les 6 URLs à utiliser

```
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/departements.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/recrutements.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/evaluations.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/corrige/outputs/vw_employes_propres.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/corrige/outputs/vw_salaires_valides.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/corrige/outputs/vw_absences_valides.csv
```

> 💡 *Remplace par ton chemin de dépôt réel. Si les CSV sont en local pour l'instant, utilise `Obtenir les données → Texte/CSV` ; le reste de la procédure est identique.*


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/powerbi/tuto/01_powerquery.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet (`reservations[date_arrivee]`, `reservations[date_depart]`, `paiements[date_paiement]`, `services[date_service]`...), c'est 4-5 tables fantômes en moins.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# II — Modéliser les données

## 2.1 Schéma en étoile — `vw_employes_propres` est la table pivot

```
                  +------------------+
                  |   Calendrier     |
                  +--------+---------+
                           | 1
                           | N
                  +--------+----------+
                  | vw_salaires_valides|
                  +--------+----------+
                           | N
                           | 1
    +------------+   1    N   +-------+----------+    1   N   +----------------+
    |departements|------------| vw_employes_propres |---------| evaluations    |
    +-----+------+            +-------+----------+            +----------------+
          |1                          | 1
          |                           | N
        N |                +----------+----------+
    +-----+------+         |  vw_absences_valides|
    |recrutements|         +---------------------+
    +------------+
```

## 2.2 Table Calendrier

```dax
Calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2021,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Mois_Nom_Long", FORMAT([Date], "mmmm", "fr-FR"),
    "Mois_Court",    FORMAT([Date], "mmm yy", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Est_Ouvre",     IF(WEEKDAY([Date], 2) < 6, 1, 0)
)
```
*La colonne `Est_Ouvre` (1 si lundi-vendredi, 0 sinon) sert au calcul du `Taux Absenteisme` qui exclut samedi/dimanche du dénominateur.*

## 2.3 Marquer Calendrier comme table de dates

Vue Données → `Calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_calendar.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 2.4 Établir les 6 relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `Calendrier` | `Date` | `vw_salaires_valides` | `periode` | Single |
| 2 | `departements` | `departement_id` | `vw_employes_propres` | `departement_id` | Single |
| 3 | `departements` | `departement_id` | `recrutements` | `departement_id` | Single |
| 4 | `vw_employes_propres` | `employe_id` | `vw_salaires_valides` | `employe_id` | Single |
| 5 | `vw_employes_propres` | `employe_id` | `vw_absences_valides` | `employe_id` | Single |
| 6 | `vw_employes_propres` | `employe_id` | `evaluations` | `employe_id` | Single |





---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les 43 mesures DAX

### Vue d'ensemble des 10 dossiers

| # | Dossier | Mesures | Rôle |
|---|---|---|---|
| 1 | KPIs de base | 5 | Effectif, Masse salariale, Salaire net, Turnover, Note perf |
| 2 | KPIs avancés | 5 | Absentéisme, Absences injustifiées, Coûts, Quadrant Performance |
| 3 | Évolution | 2 | Masse Sal Mois Précédent + Variation % |
| 4 | Sous-titres dynamiques | 5 | 1 par page |
| 5 | Cards KPI Page 1 + Performance | 10 | Sous-textes des cards, labels formatés |
| 6 | HTML Content | 4 | Effectif vs Cible, Heatmap, Coûts canaux, Quadrant scatter |
| 7 | Motifs de départ | 2 | % par motif + couleur sémantique |
| 8 | Coût Turnover par Dept | 2 | Couleur + Rang |
| 9 | Taux Turnover par Dept | 1 | Couleur sémantique selon seuils |
| 10 | Recommandations | 3 | % Recommandation + Couleur + Titre |

*Le détail DAX des 43 mesures est documenté dans le pbix lui-même (descriptions remplies, accessibles depuis l'éditeur de mesure). Voir aussi les sections 4.1 à 4.10 ci-dessous.*

## 4.1 Dossier `1. KPIs de base` (5 mesures)

```dax
Effectif Actif = 
CALCULATE(
    DISTINCTCOUNT(vw_employes_propres[employe_id]),
    vw_employes_propres[statut] = "Actif"
)

Masse Salariale Brute = 
SUMX(vw_salaires_valides, vw_salaires_valides[salaire_brut] + vw_salaires_valides[prime])

Salaire Net Moyen = AVERAGE(vw_salaires_valides[salaire_net])

Taux Turnover = 
DIVIDE(
    CALCULATE(DISTINCTCOUNT(vw_employes_propres[employe_id]), NOT(ISBLANK(vw_employes_propres[date_depart]))),
    DISTINCTCOUNT(vw_employes_propres[employe_id])
)

Note Performance Moyenne = AVERAGE(evaluations[note_globale])
```

### ⚠️ Piège — `Effectif Actif`

Si tu utilises `COUNTROWS(vw_employes_propres)` sans filtre, tu comptes les 506 employés (actifs + démissionnés). Le KPI "effectif actuel" = uniquement statut `Actif` = 446. C'est le piège n°1.

## 4.2 Dossier `2. KPIs avancés` (5 mesures)

```dax
Taux Absenteisme = 
DIVIDE(
    SUM(vw_absences_valides[duree_jours]),
    [Effectif Actif] * CALCULATE(SUM(Calendrier[Est_Ouvre]))
)

Taux Absences Injustifiees = 
DIVIDE(
    CALCULATE(SUM(vw_absences_valides[duree_jours]), vw_absences_valides[justifiee] = 0),
    SUM(vw_absences_valides[duree_jours])
)

Cout Recrutement Moyen = DIVIDE(SUM(recrutements[cout_total]), SUM(recrutements[nb_embauches]))

Cout Turnover Estime = 
VAR _nb_departs = CALCULATE(
    DISTINCTCOUNT(vw_employes_propres[employe_id]),
    NOT(ISBLANK(vw_employes_propres[date_depart]))
)
RETURN _nb_departs * [Salaire Net Moyen] * 6

Quadrant Performance = 
VAR _note = MAX(evaluations[note_globale])
VAR _salaire = MAX(vw_salaires_valides[salaire_net])
RETURN SWITCH(TRUE(),
    _note >= 3.83 && _salaire <  914943, "Top Perf Sous-Paye",
    _note >= 3.83 && _salaire >= 914943, "Top Perf Bien Paye",
    _note <  3.83 && _salaire <  914943, "A Accompagner",
    _note <  3.83 && _salaire >= 914943, "Faible Perf Bien Paye"
)
```

### 📘 Quadrant 2×2 — clé de lecture

| | Salaire < 914 943 FCFA | Salaire ≥ 914 943 FCFA |
|---|---|---|
| **Note ≥ 3,83** | **Top Perf Sous-Payé** (114 — risque départ élevé) | **Top Perf Bien Payé** (112 — RAS) |
| **Note < 3,83** | **À Accompagner** (106 — formation prioritaire) | **Faible Perf Bien Payé** (108 — sujet RH sensible) |

## 4.3 Dossiers 3 à 10 — récapitulatif des 33 mesures restantes

**3. Évolution** (2) : `Masse Sal Mois Prec` (PREVIOUSMONTH), `Variation Masse Salariale` (DIVIDE delta vs mois précédent).

**4. Sous-titres dynamiques** (5) : 1 mesure par page combinant slicers actifs (département, année, mois) avec les KPIs principaux.

**5. Cards KPI Page 1 + Performance** (10) : Sous-textes contextuels ("60 départs / 506 total"), labels formatés ("313 M"), titres dynamiques ("VARIATION JUIN vs MAI 2024").

**6. HTML Content** (4) : `Effectif Actif vs Cible HTML`, `Heatmap Absences HTML`, `Cout Embauche Canal HTML`, `Scatter Quadrant HTML` — tous avec le pattern `CONCATENATEX + style inline`.

**7. Motifs de départ** (2) : `% Motif Depart` + `Couleur Motif Depart` (gris Mutation, marine Fin CDD/Retraite, orange Démission, rouge Licenciement).

**8. Coût Turnover par Dept** (2) : `Cout Turnover Dept Rang` (RANKX) + `Couleur Cout Turnover Dept` (top 3 marine, rang 4 rouge, autres gris).

**9. Taux Turnover par Dept** (1) : `Couleur Taux Turnover Dept` (rouge ≥ 15 %, orange 10-15 %, vert < 10 %).

**10. Recommandations** (3) : `% Recommandation` (filtre dynamique année), `Couleur Recommandation` (par catégorie), `Titre Recommandations`.

---
# V — Design system

## 5.1 Charte TelcomCI — Bleu Marine + Orange

| Rôle | Hex | Usage |
|---|---|---|
| Sidebar foncée | `#1F2547` | Bandeau de navigation |
| Primaire (marine) | `#1F2547` | Titres, courbes, barres dominantes |
| Accent (orange) | `#EF9F27` | Logo, item actif sidebar, cible, alerte modérée |
| Vert | `#1D9E75` | Sain, augmentation, taux turnover < 10 % |
| Rouge | `#E24B4A` | Alerte critique, taux > 15 %, licenciement |
| Gris bleuté | `#7891B5` | Statu quo, motifs neutres |
| Fond clair | `#F0F4F8` | Cards, fond page |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes. La méthode pro : dessiner dans **PowerPoint** (mockup vierge sans données), exporter en **PNG haute résolution** (1280×720), importer comme **arrière-plan de page**, poser les visuels Power BI **par-dessus**.

### 🔧 Méthode 1 — Export PNG depuis PowerPoint à 150 DPI

1. **Win + R** → `regedit` → **Entrée**
2. Aller dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
3. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`, Valeur : `150`
4. Redémarrer PowerPoint, **Fichier → Enregistrer sous → PNG → Toutes les diapositives**

### 🔧 Méthode 2 — CloudConvert

[cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png) → upload `mockup_telcomci_blank.pptx` → 150 DPI → 1280×720.

### Renommage final

```
bg-00-cover.png
bg-01-vue-drh.png
bg-02-masse-salariale.png
bg-03-turnover.png
bg-04-absenteisme.png
bg-05-performance.png
```

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page** (icône pinceau)
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement** → **Adapter** · **Transparence** → **0 %**

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/_blank.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>



---
# VI — Construire les 6 pages

Cette partie détaille **chaque visuel** avec sa configuration exacte (type, axes, couleurs, étiquettes) et les **méthodes Power BI** non-triviales nécessaires pour le rendu final.

## 6.0 Page 0 — Couverture

> *Page d'accueil avec les 6 chiffres clés.*

**1. Bandeau tag pill « DataProjectLab · TelcomCI RH Analytics »**

- Type : Zone de texte (rectangle plein orange `#EF9F27`)
- Texte : « DataProjectLab · TelcomCI RH Analytics » blanc Segoe UI 11pt
- Coins arrondis : 6px
- Position : haut-gauche, hauteur 30px, largeur ~250px

**2. Titre principal « RH & Masse Salariale Analytics »**

- Type : Zone de texte sur 2 lignes
- Ligne 1 « RH & Masse » : Segoe UI Semibold 60pt, blanc `#FFFFFF`
- Ligne 2 « Salariale Analytics » : Segoe UI Semibold 60pt, orange `#EF9F27`
- Filet horizontal orange 2px sous le titre

**3. 6 KPIs hero en bas de page**

- Type : 6 Cartes (sans bordure ni fond)
- Valeurs : `[Effectif Actif]` 446, `[Taux Turnover Label]` 11.9%, `[Note Performance Moyenne]` 3,83/5, `[Taux Absences Injustifiees]` 17,5%, `[Cout Recrutement Moyen]` formaté en kFCFA, `[Cout Turnover Estime]` formaté en M
- Police valeur : Segoe UI Bold 32pt blanc
- Police label : Segoe UI 9pt UPPERCASE gris bleuté `#7891B5`
- Séparateurs : trait vertical 1px gris entre chaque carte

> 🎓 **METHODE — Comment formater "1 366K F" en card simple ?**
>
> Crée une mesure intermédiaire `Cout Recrutement Label = FORMAT(ROUND([Cout Recrutement Moyen]/1000, 0), "#,0") & "K F"`. Power BI ne sait pas appliquer un format custom "K F" sur une mesure numérique simple — il faut passer par une mesure texte qui renvoie directement la string finale.

**4. Sidebar de navigation footer**

- Type : 5 Boutons natifs avec action **Navigation de page**
- Texte : `01 Vue DRH`, `02 Masse Salariale`, `03 Turnover`, `04 Absenteisme`, `05 Performance`
- Police : Segoe UI 11pt blanc transparent 60%
- Disposition : 5 colonnes équidistantes avec séparateurs trait vertical 1px gris

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/04_page_cover.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.1 Page 1 — Vue DRH

> *« Quelle est la santé RH globale ce trimestre ? »*

**1. Slicers globaux (3)**

- Slicer Année : Liste horizontale 4 boutons (`Calendrier[Annee]` 2021-2024). Format → Boutons → arrondi 4px, fond inactif marine, fond actif orange `#EF9F27`
- Slicer Mois : Liste déroulante (`Calendrier[Mois_Nom]`)
- Slicer Département : Liste déroulante (`departements[nom]`)

**2. KPI 1 — Effectif Actif**

- Type : Carte avec border-left 4px marine
- Valeur : `[Effectif Actif]` (Segoe UI Bold 32pt marine `#1F2547`)
- Sous-texte : `[Sous Texte Effectif Actif]` (Segoe UI Light 11pt gris)
- Label haut : « EFFECTIF ACTIF » UPPERCASE gris

**3. KPI 2 — Taux Turnover**

- Type : Carte avec border-left 4px rouge
- Valeur : `[Taux Turnover Label]` (32pt rouge `#E24B4A`)
- Sous-texte : `[Sous Texte Taux Turnover]` « 60 departs / 506 total »

**4. KPI 3 — Note Performance Moyenne**

- Type : Carte avec border-left 4px vert
- Valeur : `[Note Performance Moyenne]` formaté `0,00"/5"` (32pt vert `#1D9E75`)
- Sous-texte : `[Sous Texte Note Performance]` (« 2023 »)

**5. KPI 4 — Coût Turnover**

- Type : Carte avec border-left 4px orange
- Valeur : `[Cout Turnover Label]` « 324M » (32pt orange `#EF9F27`)
- Sous-texte : `[Sous Texte Cout Turnover]` « 6 mois salaire x 60 departs »

**6. Effectif actif vs cible par département**

- Type : **HTML Content** (visuel marketplace de Daniel Marsh-Patrick)
- Mesure : `[Effectif Actif vs Cible HTML]`
- Rendu : 5 lignes (top 5 départements par cible), barre marine = effectif réel, marqueur orange pointillé = position cible, ratio « 115/130 » à droite

> 🎓 **METHODE — Comment afficher "115/130" comme étiquette ?**
>
> Soit tu utilises le HTML Content (mesure `Effectif Actif vs Cible HTML` avec `CONCATENATEX` qui construit la string complète), soit tu fais une variante native : crée une mesure `Label Effectif Cible = [Effectif Actif] & "/" & SUM(departements[effectif_cible])`, puis utilise-la dans les **étiquettes de données** du visuel barre native (Format → Étiquettes de données → Personnalisé → Champ `[Label Effectif Cible]`).

**7. Donut Répartition contrats**

- Type : Anneau (Donut chart)
- Catégorie : `vw_employes_propres[type_contrat]` (CDI / CDD / Intérim)
- Valeur : `[Effectif Actif]`
- Couleurs : CDI marine `#1F2547`, CDD orange `#EF9F27`, Intérim gris `#888888`
- Étiquettes : afficher pourcentage (81,17 % / 13,9 % / 4,93 %)
- Trou central : 70 %
- Légende : à droite, format "Catégorie"

**8. Évolution masse salariale mensuelle 2023**

- Type : Ligne avec marqueurs et zone
- Axe X : `Calendrier[Mois_Court]` (filtré sur Année 2023)
- Axe Y : `[Masse Salariale Brute]`
- Marqueurs : ronds orange `#EF9F27`, taille 6
- Couleur ligne : marine `#1F2547`, épaisseur 2px
- Étiquettes de données : valeur en M FCFA (420M, 421M, ...)
- Aire sous la ligne : remplie en gris pâle `#E5E5E5` 30 % opacité

> 🎓 **METHODE — Pic de juin (542M) qui interroge ?**
>
> Le pic correspond au **versement des primes annuelles**. Pour le rendre lisible sur le visuel, ajoute une **annotation** : Format → Annotations → Add → texte « Primes annuelles », ancrer à juin 2023, couleur orange. La ligne entière reste en marine, l'annotation flotte au-dessus du pic.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/05_page_vue_drh.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Masse Salariale (Structure & Évolution)

> *« Comment évolue la paie et où se concentre-t-elle ? »*

**1. KPI 1 — Masse brute du mois sélectionné**

- Type : Carte avec border-left 4px marine
- Valeur : `[Masse Salariale Brute]` formatée en M FCFA (« 5 751M » pour juin 2024)
- Label haut : « MASSE BRUTE JUIN 2024 (FCFA) » dynamique

> 🎓 **METHODE — Formater "5 751M" en card sans format custom**
>
> Crée une mesure `Masse Brute Label = FORMAT(ROUND([Masse Salariale Brute] / 1000000, 0), "#,0") & "M"`. Le diviseur 1 000 000 et le suffixe « M » sont concaténés en string.

**2. KPI 2 — Salaire Net Moyen**

- Type : Carte avec border-left 4px vert
- Valeur : `[Salaire Net Moyen]` formaté `#,0` « 901 097 »
- Sous-texte : « Sur periode complete »

**3. KPI 3 — Variation Masse Salariale**

- Type : Carte avec border-left 4px orange
- Titre dynamique : `[Titre Variation Masse Salariale]` (« VARIATION DEC vs NOV 2023 »)
- Valeur : `[Variation Masse Salariale]` formatée `+0,0%;-0,0%;0%` (orange si positif)
- Sous-texte : `[Sous Texte Variation Masse Salariale]` (« Nov: 792,8 M -> Dec: 5751,2 M »)

**4. Masse salariale par département**

- Type : Barres horizontales
- Axe Y : `departements[nom]`
- Axe X : `[Masse Salariale Brute]`
- Couleur : dégradé marine `#1F2547` (foncé) → `#7891B5` (pâle) selon le rang
- Tri : descendant sur la valeur
- Étiquettes de données : valeur en M FCFA
- Pas d'axe X visible ("Format → Axe X → Off")

> 🎓 **METHODE — Dégradé marine → bleu pâle automatique**
>
> Format → Couleurs des données → fx → **Mise en échelle des couleurs** → choisir une mesure `[Masse Salariale Brute]`, couleur min `#7891B5`, couleur max `#1F2547`. Power BI répartit automatiquement les barres sur le dégradé selon leur valeur.

**5. Tableau Brut vs Net par département**

- Type : Table
- Colonnes : `departements[nom]`, `[Masse Brute Label]` (col M), `[Salaire Net Moyen]` (col verte)
- En-tête : marine, texte blanc UPPERCASE
- Lignes : alternance fond très pâle gris/blanc
- Tri : par défaut sur la colonne Brut DESC

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/06_page_masse_salariale.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Turnover & Départs

> *« Qui part, où, à quel coût ? »*

**1. KPI 1 — Taux Turnover**

- Type : Carte avec border-left 4px rouge
- Valeur : `[Taux Turnover Label]` « 11,9 % » (rouge `#E24B4A`)
- Sous-texte : `[Sous Texte Taux Turnover]` « 60 departs / 506 total »

**2. KPI 2 — Coût Turnover**

- Type : Carte avec border-left 4px orange
- Valeur : `[Cout Turnover Label]` « 324M » (orange)
- Sous-texte : « 6 mois salaire x 60 departs »

**3. KPI 3 — Salaire Net Moyen**

- Type : Carte avec border-left 4px marine
- Valeur : `[Salaire Net Moyen]` « 901 097 »
- Sous-texte : « Base calcul cout turnover »

**4. Taux turnover par département (avec ligne cible 15 %)**

- Type : Barres horizontales
- Axe Y : `departements[nom]`
- Axe X : `[Taux Turnover]` formaté `0,0%`
- Couleur barres : `[Couleur Taux Turnover Dept]` (rouge ≥ 15 %, orange 10-15 %, vert < 10 %)
- Tri : descendant
- Étiquettes : valeur en %

> 🎓 **METHODE — Ligne cible 15 % en pointillé rouge**
>
> Sur le visuel barres, va dans **Format → Lignes constantes → Ajouter une ligne**. Configure : Valeur = 0.15, Couleur = `#E24B4A`, Style = Pointillé, Épaisseur = 2px, Étiquette de texte = « Cible 15% » position « Au-dessus à droite ». Sans cette ligne, l'utilisateur n'a pas de référence visuelle pour juger un département.

**5. Motifs de départ**

- Type : Barres horizontales
- Axe Y : `vw_employes_propres[motif_depart]`
- Axe X : `[% Motif Depart]`
- Couleur barres : `[Couleur Motif Depart]` (gris Mutation, marine Fin CDD/Retraite, orange Démission, rouge Licenciement)
- Tri : descendant sur la valeur
- Étiquettes : pourcentage (28 %, 22 %, 18 %, 17 %, 15 %)

**6. Coût turnover estimé par dept (Top 5)**

- Type : Barres horizontales
- Axe Y : `departements[nom]`
- Axe X : `[Cout Turnover Estime]` formaté en M
- Filtre : `[Cout Turnover Dept Rang]` ≤ 5 (filtre sur ce visuel uniquement)
- Couleur barres : `[Couleur Cout Turnover Dept]` (top 3 marine, rang 4 rouge, rang 5 gris)
- Étiquettes : valeur en M (« 73M », « 64M »...)

> 🎓 **METHODE — Filtrer un visuel sur le Top N par mesure calculée**
>
> Volet **Filtres** → Filtres sur ce visuel → glisser `departements[nom]` → Type de filtre : **Top N** → Afficher : 5, Par valeur : `[Cout Turnover Estime]`. Alternative DAX : créer la mesure `[Cout Turnover Dept Rang]` puis filtrer Rang ≤ 5 (équivalent fonctionnel mais plus stable en cas d'ex æquo).

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/07_page_turnover.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — Absentéisme (Heatmap)

> *« Quels départements / mois alertent ? »*

**1. Slicer Année**

- Type : Liste horizontale 4 boutons
- Champ : `Calendrier[Annee]`
- Bouton actif : fond orange `#EF9F27`, texte blanc
- Bouton inactif : fond marine `#1F2547`, texte blanc

**2. Heatmap Absences (visuel principal)**

- Type : **HTML Content** (marketplace Daniel Marsh-Patrick)
- Mesure : `[Heatmap Absences HTML]`
- Rendu : Grille 10 départements × 12 mois (J F M A M J J A S O N D)
- Palette 5 paliers :
  - 0 : blanc `#FFFFFF`
  - 1-7 : gris pâle `#F0F0F0`
  - 8-14 : orange clair `#FCC97A`
  - 15-24 : orange `#EF9F27`
  - ≥ 25 : rouge `#E24B4A`
- Légende : barre horizontale 5 pastilles en bas du visuel

> 🎓 **METHODE — Pourquoi pas une matrice native pour la heatmap ?**
>
> La matrice native Power BI permet la mise en forme conditionnelle par cellule (Format → Cellules → fx → Couleur d'arrière-plan), mais elle gère les couleurs en **gradient continu** (min → max). Pour des **paliers discrets** (0 / 1-7 / 8-14 / 15-24 / ≥25) avec ces couleurs précises, il faut soit utiliser des règles conditionnelles complexes (5 règles à configurer), soit le HTML Content qui rend exactement ce que tu veux. Le HTML est plus rapide à maintenir.

**Variante native si HTML Content indisponible**

- Type : Matrice
- Lignes : `departements[nom]`
- Colonnes : `Calendrier[Mois_Lettre]` (J F M A M...)
- Valeurs : `SUM(vw_absences_valides[duree_jours])`
- Format → Cellules → **Couleur d'arrière-plan** → Règles :
  - Si valeur = 0 → blanc
  - Si 1 ≤ valeur ≤ 7 → gris
  - Si 8 ≤ valeur ≤ 14 → orange clair
  - Si 15 ≤ valeur ≤ 24 → orange
  - Si ≥ 25 → rouge

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/08_page_absenteisme.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.5 Page 5 — Performance & Recrutement

> *« Quels top performers sont sous-payés et risquent de partir ? »*

**1. KPI 1 — Top Perf Sous-Payés**

- Type : Carte avec border-left 4px rouge
- Valeur : `[Nb Top Perf Sous-Payes]` (114, rouge `#E24B4A`)
- Sous-texte : `[Sous Texte Top Perf Sous-Payes]` (« Risque depart eleve - 440 evalues »)
- Label haut : « TOP PERF SOUS-PAYES »

**2. KPI 2 — Alerte Critique Q1+Note≥4**

- Type : Carte avec border-left 4px rouge
- Valeur : `[Nb Alerte Critique Q1]` (35, rouge)
- Sous-texte : `[Sous Texte Alerte Critique]` (« Sal. moy. 622 403 FCFA - note 4.21 »)

**3. KPI 3 — Note Performance Moyenne**

- Type : Carte avec border-left 4px vert
- Valeur : `[Note Performance Moyenne]` formaté `0,00"/5"` (3,83/5, vert)
- Sous-texte : `[Sous Texte Note Mediane]` (« Mediane sal. net : 914 943 FCFA »)

**4. Scatter Quadrant Performance × Salaire**

- Type : **HTML Content**
- Mesure : `[Scatter Quadrant HTML]`
- Rendu : Matrice 2×2 stylisée avec valeurs figées dans chaque quadrant :
  - Haut-gauche (rouge) : Top Perf Sous-Payés = **114**
  - Haut-droite (vert) : Top Perf Bien Payés = **112**
  - Bas-gauche (orange) : À Accompagner = **106**
  - Bas-droite (gris bleuté) : Faible Perf Bien Payés = **108**
- Médianes affichées : 3.83 (note, ligne horizontale) et 914 943 F (salaire, ligne verticale)

> 🎓 **METHODE — Variante native pour le scatter Quadrant**
>
> Si tu veux un vrai scatter avec un point par employé (et non un agrégat 2×2), utilise le visuel **Nuage de points natif** :
>
> - Détails : `vw_employes_propres[employe_id]`
> - Axe X : `[Salaire Net Moyen]` (par employé via la relation)
> - Axe Y : `[Note Performance Moyenne]`
> - Légende : **colonne calculée** `vw_employes_propres[Quadrant Col]` (à créer en répliquant la logique de la mesure `Quadrant Performance` mais en colonne) — une mesure ne peut pas être une légende
> - Lignes constantes : Format → Lignes constantes → 1 ligne X à 914943 (salaire médian), 1 ligne Y à 3.83 (note médiane)

**5. Coût par embauche par canal**

- Type : **HTML Content**
- Mesure : `[Cout Embauche Canal HTML]`
- Rendu : Bar chart horizontal trié ASC par coût (Cabinet 1 714k vert leader, Référencement 1 873k vert, LinkedIn 2 195k orange, Cooptation 2 243k orange, Jobboard 2 420k rouge)
- Longueur de barre : inversement proportionnelle au coût (= économie en %)

**Variante native** : Barres horizontales `recrutements[canal]` × `[Cout Recrutement Moyen]`, couleur conditionnelle via mesure `Couleur Canal Recrutement` (vert ≤ 1.9M, orange ≤ 2.3M, rouge > 2.3M).

**6. Recommandations 2023**

- Type : Barres horizontales **natives**
- Axe Y : `evaluations[recommandation]`
- Axe X : `[% Recommandation]` formaté `0,0%`
- Couleur barres : `[Couleur Recommandation]` (Statu quo gris bleuté, Formation marine, Augmentation vert, Attention rouge, Promotion orange)
- Tri : descendant sur la valeur
- Étiquettes : pourcentage à droite de chaque barre
- Titre du visuel : `[Titre Recommandations]` ("Recommandations 2023")

> 🎓 **METHODE — Couleur barres conditionnelle par catégorie**
>
> Format → **Couleurs des données** → fx (à côté de la couleur par défaut) → **Mettre en forme par : Valeur du champ** → choisir la mesure `[Couleur Recommandation]`. Power BI utilise alors le code hex retourné par la mesure pour chaque ligne. Cette approche marche pour tous les visuels barres / colonnes et évite de créer 5 règles conditionnelles séparées.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/powerbi/tuto/09_page_performance.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Slicers globaux** : `Calendrier[Annee]` (4 boutons horizontaux), `Calendrier[Mois_Nom]` (déroulant), `departements[nom]` (déroulant). Clic droit → **Synchroniser les segments → cocher toutes les pages**.

**Navigation sidebar** : 5 boutons natifs avec **action Navigation de page**. Bouton actif fond orange `#EF9F27`, inactif fond marine sombre. Bouton « Retour à l'accueil » en bas → action vers la page Cover.

> 🎓 **METHODE — État actif de la sidebar par page**
>
> Power BI ne gère pas nativement l'état "actif" d'un bouton de navigation selon la page courante. Astuce : sur chaque page, **dupliquer le bouton actif** avec le style orange (au lieu d'avoir le même bouton sur toutes les pages). Le bouton actif n'a pas d'action Navigation (il pointe sur la page courante).

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 6 tables sources + Calendrier + _Mesures, 6 relations actives, 0 LocalDateTable · `Calendrier` marquée comme table de dates.

**Mesures** : 43 dans `_Mesures` · 10 dossiers numérotés · format défini.

**Visuels HTML Content** : 4 visuels marketplace actifs.

**Pages** : 6 pages (Cover + 5 nav) · Slicers Année/Mois/Département synchronisés · Navigation sidebar avec boutons d'action Navigation.

## 8.2 Pièges fréquents

| Symptôme | Cause | Correction |
|---|---|---|
| `Effectif Actif` retourne 506 au lieu de 446 | Pas de filtre `statut = "Actif"` | Wrapper le DISTINCTCOUNT dans CALCULATE avec le filtre |
| `Taux Absenteisme` à 0,2 % au lieu de 0,6 % | Dénominateur inclut samedi/dimanche | Utiliser `Calendrier[Est_Ouvre]` |
| `PREVIOUSMONTH` renvoie blank | `Calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| Visuel HTML Content vide | Marketplace pas installé | Installer HTML Content de Daniel Marsh-Patrick |
| Pic anormal de masse salariale en juin | Versement primes annuelles | Documenter via annotation sur le visuel |
| Ligne cible 15 % invisible | Pas configurée dans Lignes constantes | Format → Lignes constantes → 0.15 rouge pointillé |
| Couleurs barres recommandations toutes pareilles | Mise en forme conditionnelle non appliquée | Format → fx → Valeur du champ → `Couleur Recommandation` |
| Bouton actif sidebar ne change pas selon la page | Power BI ne gère pas l'état actif natif | Dupliquer le bouton avec le style actif sur chaque page |

## 8.3 Storytelling exécutif

1. **Vue DRH** : « 446 actifs sur 506. Turnover 11,9 %. Note 3,83/5. Coût turnover annuel : 324 M FCFA. »
2. **Masse Salariale** : « 5,75 Md en juin 2024 (avec primes annuelles). Concentration : Technique + Service Client + Commercial = 60 % de la masse. »
3. **Turnover** : « Marketing en zone rouge (23,3 % vs cible 15 %). Démission = 17 % des motifs. »
4. **Absentéisme** : « 17,5 % d'absences injustifiées. Service Client a explosé en septembre/octobre 2023. »
5. **Performance** : « 35 employés en alerte critique (note ≥ 4 ET salaire < médiane). Coût d'augmentation : 84 M FCFA/an. ROI : 5 départs évités → 27 M économisés. »

## 8.4 Mapping mockup PPTX ↔ pages Power BI

| Slide | Background PNG | Page |
|---|---|---|
| 0 | `bg-00-cover.png` | Couverture |
| 1 | `bg-01-vue-drh.png` | Vue DRH |
| 2 | `bg-02-masse-salariale.png` | Masse Salariale |
| 3 | `bg-03-turnover.png` | Turnover |
| 4 | `bg-04-absenteisme.png` | Absentéisme |
| 5 | `bg-05-performance.png` | Performance |

---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">TelcomCI — RH Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>